In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [4]:
import tensorflow as tf
AUTOTUNE = tf.data.AUTOTUNE
IMG = (32, 32); BATCH = 64

train_ds = tf.keras.utils.image_dataset_from_directory(
    'CIFAKE/train', label_mode='binary', image_size=IMG,
    batch_size=BATCH, validation_split=0.2, subset='training', seed=42)
val_ds = tf.keras.utils.image_dataset_from_directory(
    'CIFAKE/train', label_mode='binary', image_size=IMG,
    batch_size=BATCH, validation_split=0.2, subset='validation', seed=42)
test_ds = tf.keras.utils.image_dataset_from_directory(
    'CIFAKE/test', label_mode='binary', image_size=IMG,
    batch_size=BATCH, shuffle=False)

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

Found 100000 files belonging to 2 classes.
Using 80000 files for training.
Found 100000 files belonging to 2 classes.
Using 20000 files for validation.
Found 20000 files belonging to 2 classes.


In [5]:
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

model = Sequential()
model.add(layers.Input(shape=(32, 32, 3)))
model.add(layers.Rescaling(1./255))
model.add(layers.RandomFlip('horizontal'))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.Conv2D(256, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.25))

model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.25))

model.add(layers.Flatten())
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [6]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5,
                           restore_best_weights=True, verbose=1)
history = model.fit(train_ds, validation_data=val_ds, epochs=40,
                    callbacks=[early_stop])

Epoch 1/40


/Users/lievwurtzel/Projects/AI Class/.venv12/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


 730/1250 ━━━━━━━━━━━━━━━━━━━━ 3:36 416ms/step - accuracy: 0.8030 - loss: 0.4278

KeyboardInterrupt: 

In [ ]:
score = model.evaluate(test_generator, verbose=1)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.8594 - loss: 0.3286
Test loss: 0.32860639691352844
Test accuracy: 0.8593999743461609
